# SOV33 Simple Training - No Unsloth

Train sovereign AI model using standard PyTorch + Transformers
Works on any GPU (P100, T4, etc.)

**Model:** Qwen/Qwen2.5-0.5B-Instruct
**GPU:** Any CUDA GPU
**Training:** SFT with standard Transformers

In [ ]:
# Cell 1: Install dependencies
!pip install -q torch==2.5.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers datasets accelerate

In [ ]:
# Cell 2: Imports
import torch
import json
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 3: Training Data
TRAINING_DATA = [
    {"prompt": "What is the DEFONEOS care floor value?", "completion": "The DEFONEOS care floor value is 0.95. This is the minimum quality threshold that all outputs must meet."},
    {"prompt": "How many agents are in the BFT council?", "completion": "The BFT council consists of 33 agents. A quorum of 23 out of 33 (23/33) is required for binding decisions."},
    {"prompt": "What cryptographic algorithm does SIGIL use?", "completion": "SIGIL uses Ed25519 cryptographic signatures. Each signature is hash-linked to the previous one, creating an immutable audit chain."},
    {"prompt": "What is Article 0 in the sovereign framework?", "completion": "Article 0 is fee-for-service only. It prohibits equity stakes, board seats, and VC money."},
    {"prompt": "When does EU AI Act Article 50 take effect?", "completion": "EU AI Act Article 50 takes effect on 2 August 2026. It covers transparency obligations for AI systems."},
    {"prompt": "What is the GDPR breach notification deadline?", "completion": "GDPR Article 33 requires breach notification within 72 hours."},
    {"prompt": "What is the GDPR maximum fine?", "completion": "GDPR Article 83 sets the maximum fine at 20 million euros or 4% of worldwide annual turnover."},
    {"prompt": "What is ISO 42001?", "completion": "ISO 42001 is the AI Management System (AIMS) standard with 7 clauses (4-10) and Annex A."},
    {"prompt": "What is AUKUS Pillar 2?", "completion": "AUKUS Pillar 2 is a 2.4 billion dollar program over 5 years focused on AI, autonomy, quantum, and cyber."},
    {"prompt": "List the 12 Pillars of the sovereign framework.", "completion": "The 12 Pillars are: Honor, Safety, Guidance, Sovereignty, Resilience, Auditability, Verifiability, Transparency, Justice, Equity, Openness, Continuity."},
    {"prompt": "What are the 7 Red Lines?", "completion": "The 7 Red Lines are: No kinetic targeting, no surveillance, no civilian harm, no sovereignty violations, no auto-escalation, no lying, no irreversibility."},
    {"prompt": "What are the 5 OWEM groups?", "completion": "The 5 OWEM groups are: compliance, defense, intuition, voice, and general."},
    {"prompt": "What is the sovereign DID?", "completion": "The sovereign DID is did:csoai:nicholas-001."},
    {"prompt": "What is the CSOAI Companies House number?", "completion": "CSOAI Ltd is registered at UK Companies House with number 16939677."},
    {"prompt": "A bat and a ball cost $1.10. The bat costs $1.00 more than the ball. How much does the ball cost?", "completion": "Let B = ball cost. Then B + (B + 1.00) = 1.10. So 2B = 0.10, B = 0.05. The ball costs $0.05."},
    {"prompt": "Can you catch a cold from being cold?", "completion": "No. Colds are caused by viruses (rhinovirus), not by cold temperatures."},
    {"prompt": "Write a Python function is_palindrome(s).", "completion": "def is_palindrome(s):\n    s = s.lower().strip()\n    return s == s[::-1]"},
    {"prompt": "What is 15% of 200?", "completion": "15% of 200 = 0.15 x 200 = 30."},
    {"prompt": "What is 2^10?", "completion": "2^10 = 1024."},
    {"prompt": "What is 7! (7 factorial)?", "completion": "7! = 7 x 6 x 5 x 4 x 3 x 2 x 1 = 5040."},
]

print(f"Training examples: {len(TRAINING_DATA)}")

In [ ]:
# Cell 4: Format data
formatted_data = []
for item in TRAINING_DATA:
    formatted_data.append({
        "messages": [
            {"role": "system", "content": "You are SOV33, a sovereign AI model. Answer concisely and accurately."},
            {"role": "user", "content": item["prompt"]},
            {"role": "assistant", "content": item["completion"]},
        ]
    })

dataset = Dataset.from_list(formatted_data)
print(f"Dataset size: {len(dataset)}")

In [ ]:
# Cell 5: Load model
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
print(f"Loading model: {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)
print("Model loaded!")

In [ ]:
# Cell 6: Tokenize dataset
def tokenize_function(examples):
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    tokenized = tokenizer(texts, truncation=True, padding=True, max_length=256)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)
print(f"Tokenized dataset: {len(tokenized_dataset)} examples")

In [ ]:
# Cell 7: Training
training_args = TrainingArguments(
    output_dir="sov-simple-output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=10,
    logging_steps=5,
    save_steps=50,
    save_total_limit=2,
    fp16=False,
    optim="adamw_torch",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()
print("Training complete!")

In [ ]:
# Cell 8: Save model
print("Saving model...")
model.save_pretrained("sov-simple-output")
tokenizer.save_pretrained("sov-simple-output")
print("Model saved to sov-simple-output/")

In [ ]:
# Cell 9: Test
print("\n=== TESTING TRAINED MODEL ===")

test_questions = [
    "What is the DEFONEOS care floor value?",
    "How many agents are in the BFT council?",
    "Can you catch a cold from being cold?",
]

for q in test_questions:
    messages = [
        {"role": "system", "content": "You are SOV33, a sovereign AI model. Answer concisely and accurately."},
        {"role": "user", "content": q},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = inputs.to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=128, do_sample=False)
    answer = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print(f"\nQ: {q}")
    print(f"A: {answer[:200]}")